# Topic modeling · versi review
**Pilot satu kasus: EDA angka → preprocessing yang diuji ablasi → 5 model topik + baseline acak → post-processing → evaluasi kelemahan.**

Run All di **Kaggle GPU**. Semua angka berasal dari data asli, tanpa grafik otomatis.

| Model | Representasi | Kata kunci topik dari | Cara kerja |
|---|---|---|---|
| Acak | kata acak dari kosakata | — | batas bawah coherence |
| TF-IDF + KMeans | BoW agresif | c-TF-IDF | baseline klasik: satu dokumen satu topik |
| LDA | hitungan kata | distribusi kata topik | probabilistik, dokumen bercampur topik |
| NMF | TF-IDF | matriks H | faktorisasi, biasanya lebih tajam dari LDA untuk teks pendek |
| e5 + KMeans | embedding e5 frozen | c-TF-IDF | semantik, satu dokumen satu topik |
| e5 + UMAP + HDBSCAN (gaya BERTopic) | embedding e5 frozen | c-TF-IDF | jumlah topik otomatis, ada outlier |

**Kriteria tanpa label** (tidak memakai kategori):
- **NPMI coherence:** apakah kata kunci topik muncul bersama di dokumen.
- **Diversity:** porsi kata unik di top-25 semua topik, yang menunjukkan apakah topik saling berbeda.
- **Topic quality** (TQ) = NPMI ternormalisasi × diversity, mengikuti Dieng dkk. 2020.
- **Stabilitas:** kemiripan kata kunci antar 3 seed.

Ditambah larangan topik raksasa. C_V coherence (gensim) dihitung untuk kandidat teratas.

**Data:** `DATASET="news"` memakai judul berita, dengan kategori sebagai audit yang baru dibuka di bagian 10. `DATASET="mbg"` memakai komentar YouTube MBG tanpa label, yaitu kondisi seperti lomba.

## 00 · Konfigurasi

Parameter papermill. Untuk data lain, tambah preset: nama file, kolom teks, dan kolom audit (`None` kalau tidak ada).

In [1]:
DATASET = "news"
SEED, N_ROWS, TOP = 42, 5000, 10
K_GRID = [5, 8, 10, 12, 15, 20]
SEEDS = 3                              # stabilitas: kata kunci dibandingkan antar 3 seed
MAX_SHARE, MAX_OUTLIER = .40, .30      # larangan topik raksasa dan outlier HDBSCAN berlebihan
MIN_DF, DF_MAX, MIN_TOPIC_SHARE, MERGE_JACCARD = 3, .25, .01, .30
MAX_LEN, BATCH = 128, 64
PRESETS = {"news": dict(file="indonesian-news-title.csv", text="title", audit="category"),
           "mbg": dict(file="comment_sentiment.csv", text="Comment", audit=None)}

In [2]:
# Parameters
DATASET = "mbg"


In [3]:
from pathlib import Path
FILE, TEXT_COL, AUDIT_COL = PRESETS[DATASET]["file"], PRESETS[DATASET]["text"], PRESETS[DATASET]["audit"]
OUT = Path(f"/kaggle/working/hasil_topik_{DATASET}")
print("dataset:", DATASET, "| teks:", TEXT_COL, "| audit:", AUDIT_COL or "tidak ada")

dataset: mbg | teks: Comment | audit: tidak ada


## 01 · Lingkungan Kaggle

Paket tambahan dipasang di Kaggle. Model e5 diambil dari paket offline kalau tersedia, dan dari Hugging Face kalau tidak. Backbone e5 di-freeze dan hanya dipakai untuk feature extraction.

In [4]:
import sys, os, subprocess
assert Path("/kaggle/input").exists(), "Jalankan notebook ini di Kaggle."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn>=0.5,<0.6", "gensim>=4.4,<5", "ftfy>=6.3,<7",
                "emoji>=2.14,<3", "datasketch>=1.6,<3", "langid==1.1.6", "Sastrawi==1.0.1"], check=True)
os.environ.update(TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1", USE_TF="0", USE_FLAX="0")

In [5]:
import json, time, random, re, html, unicodedata, itertools, hashlib, warnings
from collections import Counter
import numpy as np, pandas as pd, torch, emoji, ftfy
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import normalize
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT.mkdir(exist_ok=True); INPUT = Path("/kaggle/input"); RUN_START = time.perf_counter()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada")

def show(title, table):
    print("\n" + title); display((table.to_frame() if isinstance(table, pd.Series) else table).round(3))
def distribution(values):
    n = values.value_counts(dropna=False); return pd.DataFrame({"n": n, "%": n / (len(values) or np.nan) * 100})
pd.set_option("display.max_columns", None, "display.max_colwidth", 110, "display.width", 200)

GPU: Tesla T4


## 02 · Muat data mentah & audit sebelum drop

Missing dan teks invalid diaudit sebelum dibuang. Kolom audit langsung dipisah ke `_audit`, dan baru dipakai di bagian 10.

In [6]:
path = next(INPUT.rglob(FILE)); raw = pd.read_csv(path)
assert TEXT_COL in raw, f"Kolom tersedia: {raw.columns.tolist()}"
show("Skema sebelum pembersihan", pd.DataFrame({"tipe": raw.dtypes.astype(str), "missing": raw.isna().sum(),
     "missing_%": raw.isna().mean() * 100, "unik_non_null": raw.nunique(dropna=True)}))
text = raw[TEXT_COL]
quality = pd.DataFrame({"teks_null": text.isna(), "teks_non_string": text.notna() & ~text.map(lambda x: isinstance(x, str)),
    "teks_kosong": text.fillna("").astype(str).str.strip().eq(""),
    "teks_pseudo_null": text.fillna("").astype(str).str.strip().str.casefold().isin(["nan", "null", "none", "n/a", "-"])})
if AUDIT_COL: quality["audit_null"] = raw[AUDIT_COL].isna()
show("Kualitas input · flag dapat overlap", pd.DataFrame({"n": quality.sum(), "%": quality.mean() * 100}))
data = raw.loc[~quality.any(axis=1)].reset_index(names="row_id")
if N_ROWS and len(data) > N_ROWS: data = data.sample(N_ROWS, random_state=SEED).reset_index(drop=True)
data["_audit"] = data[AUDIT_COL].astype(str) if AUDIT_COL else None
print("File:", path.name, "| baris mentah:", len(raw), "| invalid:", int(quality.any(axis=1).sum()), "| baris pilot:", len(data))


Skema sebelum pembersihan


,tipe,missing,missing_%,unik_non_null
Comment ID,object,0,0.0,5034
Comment Link,object,0,0.0,5034
Commenter Name,object,0,0.0,4431
Commenter ID,object,0,0.0,4431
Comment,object,0,0.0,4996
Likes,int64,0,0.0,45
Cleaned Comment,object,0,0.0,4996
Sentiment,object,0,0.0,3
Confidence Score,float64,0,0.0,4966



Kualitas input · flag dapat overlap


,n,%
teks_null,0,0.0
teks_non_string,0,0.0
teks_kosong,0,0.0
teks_pseudo_null,0,0.0


File: comment_sentiment.csv | baris mentah: 5035 | invalid: 0 | baris pilot: 5000


## 03 · EDA wajib topic modeling · angka

Pertanyaan EDA untuk topic modeling: apakah dokumennya cukup panjang (LDA butuh banyak kata per dokumen), apakah kosakatanya cukup padat setelah pruning, kata apa yang ada di mana-mana (akan muncul di semua topik), frasa dua kata apa yang sering muncul (kandidat bigram), noise apa yang perlu dibuang, dan apakah teks identik atau template akan membentuk topik palsu. Setiap angka menghasilkan keputusan di 3.6.

In [7]:
t = data[TEXT_COL].astype(str)
key = t.str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
words = t.str.split().str.len()
show("3.1 Ukuran, duplikat, panjang dokumen", pd.Series({"dokumen": len(t), "duplikat_normal": key.duplicated().sum(),
    "duplikat_normal_%": 100 * key.duplicated().mean(), "grup_duplikat": key.value_counts().gt(1).sum(),
    "dok_<5_kata_%": 100 * words.lt(5).mean(), "dok_<10_kata_%": 100 * words.lt(10).mean(), "dok_>100_kata_%": 100 * words.gt(100).mean()}))
show("Panjang kata · LDA lemah di dokumen sangat pendek", words.describe(percentiles=[.05, .25, .5, .75, .95, .99]))


3.1 Ukuran, duplikat, panjang dokumen


,0
dokumen,5000.00
duplikat_normal,45.00
duplikat_normal_%,0.90
grup_duplikat,31.00
dok_<5_kata_%,10.46
dok_<10_kata_%,27.34
dok_>100_kata_%,4.04



Panjang kata · LDA lemah di dokumen sangat pendek


,Comment
count,5000.000
mean,28.930
std,48.176
min,1.000
5%,3.000
25%,9.000
50%,17.000
75%,33.000
95%,87.000
99%,187.070


In [8]:
seqs = t.str.casefold().str.findall(r"[^\W\d_]+")
counts = pd.Series(Counter(w for s in seqs for w in s), dtype="int64").sort_values(ascending=False)
docfreq = pd.Series(Counter(w for s in seqs for w in set(s)), dtype="int64").reindex(counts.index) / len(seqs)
kept = docfreq[(docfreq * len(seqs) >= MIN_DF) & (docfreq <= DF_MAX)]
show("3.2 Kosakata & pruning", pd.Series({"token": counts.sum(), "tipe_unik": len(counts), "hapax_%_tipe": 100 * counts.eq(1).mean(),
    "hapax_%_token": 100 * counts.eq(1).sum() / counts.sum(), f"tipe_df>={MIN_DF}": int((docfreq * len(seqs) >= MIN_DF).sum()),
    f"kata_df>{DF_MAX:.0%}": int(docfreq.gt(DF_MAX).sum()), "kosakata_setelah_pruning": len(kept),
    "token_tersisa_%": 100 * counts.reindex(kept.index).sum() / counts.sum(),
    "dok_kosong_setelah_pruning_%": 100 * seqs.map(lambda s: not any(w in kept.index for w in s)).mean()}))
show("Top words · kandidat stopword umum/domain", pd.DataFrame({"n": counts.head(TOP + 5), "%_dokumen": 100 * docfreq.head(TOP + 5)}))


3.2 Kosakata & pruning


,0
token,150817.000
tipe_unik,13987.000
hapax_%_tipe,53.164
hapax_%_token,4.930
tipe_df>=3,3443.000
kata_df>25%,4.000
kosakata_setelah_pruning,3439.000
token_tersisa_%,81.770
dok_kosong_setelah_pruning_%,1.480



Top words · kandidat stopword umum/domain


,n,%_dokumen
mbg,3648,45.80
br,3051,16.22
di,2952,32.02
dan,2910,30.20
yg,2696,28.68
yang,2066,19.68
itu,1403,18.92
ini,1381,18.30
ada,1209,16.74
tidak,1175,15.78


In [9]:
pair_counts = Counter(" ".join(s[i:i + 2]) for s in seqs for i in range(len(s) - 1))
total_tokens = counts.sum()
collocation = pd.DataFrame([{"bigram": b, "n": n, "PMI": np.log(n * total_tokens / (counts[b.split()[0]] * counts[b.split()[1]]))}
                            for b, n in pair_counts.items() if n >= 10])
show("3.3 Kolokasi · bigram sering dan PMI tinggi, kandidat frasa untuk topik", collocation.sort_values("PMI", ascending=False).head(TOP) if len(collocation) else collocation)
NOISE = {"url": r"https?://|www\.", "html": r"</?[a-zA-Z][^>]*>", "html_entity": r"&(?:#\d+|[a-zA-Z]+);", "mention": r"@\w+",
    "hashtag": r"#\w+", "emoji": r"[\U0001F300-\U0001FAFF☀-➿]", "angka": r"\d", "huruf_ulang": r"([^\W\d_])\1{2,}",
    "kapital_semua": r"\b[A-Z]{3,}\b", "non_ASCII": r"[^\x00-\x7f]", "slang_contoh": r"\b(?:gk|gak|ga|bgt|yg|utk|udh|klo|jg|tdk)\b",
    "tawa": r"\b(?:wk){2,}\w*|\b(?:ha){2,}\b"}
flags = pd.DataFrame({n: t.str.contains(p, flags=0 if n == "kapital_semua" else re.I) for n, p in NOISE.items()})
show("3.4 Variasi noise · % dokumen", pd.DataFrame({"n": flags.sum(), "%": flags.mean() * 100}).sort_values("n", ascending=False))


3.3 Kolokasi · bigram sering dan PMI tinggi, kandidat frasa untuk topik


,bigram,n,PMI
73,dv ujg,16,9.151
69,uckszu wh,16,9.151
70,wh gy,16,9.151
72,mb dv,16,9.091
71,gy mb,16,9.091
1079,attack rate,17,9.033
350,h o,17,8.979
968,search query,22,8.833
967,results search,22,8.833
659,nye nye,16,8.607



3.4 Variasi noise · % dokumen


,n,%
kapital_semua,2160,43.20
slang_contoh,1975,39.50
angka,1541,30.82
non_ASCII,902,18.04
html,900,18.00
emoji,791,15.82
html_entity,400,8.00
huruf_ulang,310,6.20
url,98,1.96
hashtag,60,1.20


In [10]:
from langid.langid import LanguageIdentifier, model as language_model
identifier = LanguageIdentifier.from_modelstring(language_model, norm_probs=True)
language = t.sample(min(500, len(t)), random_state=SEED).map(lambda s: identifier.classify(s[:2000]))
show("3.5 Estimasi bahasa · sampel 500, yakin >= 0.8", distribution(language.map(lambda p: p[0] if p[1] >= .8 else "tidak_pasti")).head(8))
prefix = seqs.map(lambda s: " ".join(s[:3])); top_prefix = prefix[prefix.str.count(" ").ge(2)].value_counts().head(5)
show("Template · awalan 3 kata tersering", top_prefix.to_frame("n").assign(**{"%": 100 * top_prefix / len(t)}))
if AUDIT_COL: show("Kategori audit · hanya untuk penilaian", distribution(data["_audit"]))
decisions = pd.DataFrame([
    {"temuan": "dok < 10 kata", "angka": f"{words.lt(10).mean():.1%}", "keputusan": "LDA kemungkinan lemah; model satu-topik-per-dokumen (KMeans, HDBSCAN) ikut diuji" if words.lt(10).mean() > .5 else "LDA layak"},
    {"temuan": f"kata df > {DF_MAX:.0%}", "angka": str(int(docfreq.gt(DF_MAX).sum())), "keputusan": "dibuang sebagai stopword domain di BoW"},
    {"temuan": "token tersisa setelah pruning", "angka": f"{counts.reindex(kept.index).sum() / counts.sum():.1%}", "keputusan": "stopword dan kata langka dibuang; stemming diuji di 6.1"},
    {"temuan": "kolokasi PMI tinggi", "angka": str(int((collocation.PMI > 5).sum()) if len(collocation) else 0), "keputusan": "bigram (gensim Phrases) diuji di 6.1"},
    {"temuan": "duplikat normal", "angka": f"{key.duplicated().mean():.1%}", "keputusan": "duplikat bisa membentuk topik palsu; dicek di 10.4"},
    {"temuan": "noise tertinggi", "angka": f"{flags.mean().idxmax()} {flags.mean().max():.1%}", "keputusan": "safe cleaning + normalisasi untuk BoW"}])
show("3.6 Temuan → keputusan", decisions)


3.5 Estimasi bahasa · sampel 500, yakin >= 0.8


,n,%
Comment,,
id,276,55.2
tidak_pasti,168,33.6
ms,36,7.2
en,5,1.0
xh,3,0.6
eo,1,0.2
nl,1,0.2
sw,1,0.2



Template · awalan 3 kata tersering


,n,%
Comment,,
a href https,58,1.16
mbg itu bagus,15,0.30
program mbg bagus,11,0.22
saya setuju mbg,10,0.20
mantap prof gema,9,0.18



3.6 Temuan → keputusan


,temuan,angka,keputusan
0,dok < 10 kata,27.3%,LDA layak
1,kata df > 25%,4,dibuang sebagai stopword domain di BoW
2,token tersisa setelah pruning,81.8%,stopword dan kata langka dibuang; stemming diuji di 6.1
3,kolokasi PMI tinggi,132,bigram (gensim Phrases) diuji di 6.1
4,duplikat normal,0.9%,duplikat bisa membentuk topik palsu; dicek di 10.4
5,noise tertinggi,kapital_semua 43.2%,safe cleaning + normalisasi untuk BoW


## 04 · Preprocessing dari universal

Fungsi berikut diambil langsung dari `preprocessing_universal/stepwise.py`. Topic model berbasis kata (LDA, NMF, KMeans TF-IDF) butuh BoW agresif: kata umum, angka, dan tanda baca dibuang, karena kata seperti itu akan muncul di semua topik. Model embedding butuh kalimat utuh.

In [11]:
SLANG = dict(zip(
    'gk ga gak nggak ngga tdk blm jgn yg dgn utk krn karna bgt tp udh sdh skrg gmn klo kalo jg bgs'.split(),
    'tidak tidak tidak tidak tidak tidak belum jangan yang dengan untuk karena karena banget tapi sudah sudah sekarang bagaimana kalau kalau juga bagus'.split()))
PROTECTED = set('mbg sppg bgn apbn apbd bpom bpk umkm allah massa jakarta bandung'.split())
NEGATIONS = set('tidak tak bukan belum jangan tanpa kurang nggak gak ga gk ora teu indak'.split())
AMBIGUOUS = set('sm dr bs pd lg org'.split())
VOCAB = set('bagus banget lama makanan makan bergizi gratis program keracunan anggaran korupsi sehat sekolah anak siswa pemerintah merata enak'.split()) | set(SLANG.values()) | PROTECTED
STOPWORDS = set('yang dan di ke dari untuk pada dengan ini itu atau juga saya kami kita mereka dia oleh sebagai dalam para serta adalah sebuah'.split()) - NEGATIONS - PROTECTED
REGIONAL = {'ora': 'tidak', 'teu': 'tidak', 'indak': 'tidak', 'apik': 'bagus'}
OBFUSCATION = {'k3r4cunan': 'keracunan', 'k3r4cun4n': 'keracunan',
               'ker@cunan': 'keracunan', 'koru*si': 'korupsi'}
# URL/email/account/hashtag are single tokens: normalization never enters them.
TOKEN = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+|[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}|(?<!\w)[@#][\w.]+|[^\W\d_]+', re.I)
WORD = re.compile(r'[^\W\d_]+')
URL = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+', re.I)


def map_words(text, transform):
    return TOKEN.sub(lambda m: transform(m[0]) if WORD.fullmatch(m[0]) else m[0], text)

In [12]:
def valid_text(value):
    # None/NaN/pd.NA/numbers/list are not converted to literal "nan"/"123".
    return value if isinstance(value, str) else ''

def repair_encoding(text):
    return ftfy.fix_text(text, unescape_html=False, uncurl_quotes=False,
                        fix_latin_ligatures=False, fix_character_width=False,
                        remove_control_chars=False, normalization=None)

def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)  # retain emoji/local scripts/fullwidth

def remove_html(text):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style']):
        tag.decompose()
    return soup.get_text(' ')

def decode_html_entities(text):
    return html.unescape(text)

def remove_invisible(text):
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    return re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

In [13]:
def replace_urls(text):
    # Preserve terminal sentence punctuation; a heuristic, not a complete URL parser.
    return URL.sub(lambda m: ' URL ' + m[0][len(m[0].rstrip('.,!?;)')):], text)

def replace_emails(text):
    return re.sub(r'\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b', ' EMAIL ', text)

def replace_phones(text):
    return re.sub(r'(?<!\w)(?:\+62|62|0)[ -]?8(?:[ -]?\d){8,11}(?!\w)', ' PHONE ', text)

def replace_mentions(text):
    return re.sub(r'(?<![\w@])@[\w.]+', ' USER ', text)

def segment_hashtags(text, mapping=None):
    mapping = {} if mapping is None else mapping
    def split(match):
        word = match[1]
        # #tolakmbg stays tolakmbg unless an explicit reviewed map is supplied.
        return ' ' + mapping.get(word.casefold(), re.sub(
            r'(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word)) + ' '
    return re.sub(r'(?<!\w)#(\w+)', split, text)

def lowercase(text):
    return text.casefold()

In [14]:
def normalize_slang(text, mapping=SLANG):
    return map_words(text, lambda w: mapping.get(w.casefold(), w)
                     if w.casefold() not in PROTECTED | AMBIGUOUS else w)

def normalize_elongation(text, vocab=VOCAB):
    def shorten(word):
        if word.casefold() in PROTECTED:
            return word
        candidates = [re.sub(r'([^\W\d_])\1{2,}', replacement, word, flags=re.I)
                      for replacement in (r'\1', r'\1\1')]
        return next((w for w in candidates if w.casefold() in vocab), word)
    return map_words(text, shorten)

def normalize_laughter(text):
    def replace(word):
        return 'wkwk' if re.fullmatch(r'(?:wk){2,}|w+k+w+k+[wk]*', word, re.I) else (
            'haha' if re.fullmatch(r'(?:ha){2,}h?', word, re.I) else word)
    return map_words(text, replace)

def normalize_reduplication(text, mapping=None):
    # A whitelist avoids turning grade S2, model GPT2, dates or quantities into words.
    mapping = {'anak2': 'anak-anak', 'orang2': 'orang-orang', 'teman2': 'teman-teman'} if mapping is None else mapping
    return re.sub(r'\b\w+2\b', lambda m: mapping.get(m[0].casefold(), m[0]), text)

In [15]:
def remove_emoji(text):
    return emoji.replace_emoji(text, replace=' ')

def remove_emoticons(text):
    return re.sub(r"(?<!\w)(?:[:;=8][\-^']?[)(/DPp]+|<3)(?!\w)", ' ', text)

def remove_numbers(text):
    # Opt-in destructive view: amounts/dates become incomplete, inspect before using.
    return re.sub(r'(?<!\w)\d+(?:[.,:/-]\d+)*(?!\w)', ' ', text)

def remove_punctuation(text):
    # Keep all scripts/letters/digits; punctuation becomes a SPACE, never concatenation.
    return re.sub(r'[^\w\s]|_', ' ', text)

def reduce_repeated_words(text):
    # Aggressive only: even "bagus bagus" can be expressive, so retain another view.
    return re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text, flags=re.I)

**4.1 Versi teks.** `embed` dipakai e5. `normal` = lowercase dan normalisasi kata, tanpa pembuangan. `bow` = normal tanpa emoji, angka, tanda baca, stopword umum, dan stopword domain. `bow_stem` = bow + stemming Sastrawi. `bow_bigram` = bow dengan frasa dua kata dari gensim Phrases, misalnya `virus_corona`. Ketiga versi BoW diuji di 6.1.

In [16]:
def stem_series(series):
    """Stem hanya kata yang lolos MIN_DF, 4 proses forkserver (aman walau CUDA sudah aktif)."""
    import multiprocessing as mp
    Path("stem_worker.py").write_text("from Sastrawi.Stemmer.StemmerFactory import StemmerFactory\n_s = None\n"
        "def stem(w):\n    global _s\n    _s = _s or StemmerFactory().create_stemmer()\n    return _s.stem(w)\n")
    if "." not in sys.path: sys.path.insert(0, ".")
    import stem_worker
    vocab = pd.Series(Counter(w for s in series.str.split() for w in set(s)))
    frequent = [w for w in vocab[vocab >= MIN_DF].index if w not in PROTECTED | NEGATIONS]
    with mp.get_context("forkserver").Pool(4) as pool:
        stems = dict(zip(frequent, pool.map(stem_worker.stem, frequent, chunksize=256)))
    print(f"stemming {len(frequent)} kata dari {len(vocab)} kata unik")
    return series.map(lambda x: " ".join(stems.get(w, w) for w in x.split()))
def chain(text, steps):
    for step in steps: text = step(text)
    return text
def apply(series, steps):
    cache = {s: chain(s, steps) for s in series.unique()}; return series.map(cache)
SAFE = [valid_text, repair_encoding, normalize_unicode, lambda s: remove_html(s) if "<" in s else s,
        decode_html_entities, remove_invisible, normalize_whitespace]
PLACEHOLDER = [replace_urls, replace_emails, replace_phones, replace_mentions, normalize_whitespace]
NORMAL = [segment_hashtags, lowercase, normalize_slang, normalize_elongation, normalize_laughter, normalize_reduplication]
CLEAN = [remove_emoji, remove_emoticons, remove_numbers, remove_punctuation, reduce_repeated_words, normalize_whitespace]
STOP = STOPWORDS | set("""akan ada adalah agar aja aku apa atau bahwa banyak baru bila bisa buat bukan cuma dah dalam dapat demi dgn
    dong gitu harus hal hanya ini itu jadi jika juga kalau kami kamu karena kata ke kenapa kok kini lagi lah lain lalu lebih
    loh maka masih mau mereka meski mungkin nah namun nanti nya oleh pada para pernah pun saat saja sama sampai sangat
    sebagai sedang sejak sekarang semua seperti setelah sih sudah supaya tapi telah tentang terhadap tersebut tetapi
    untuk usai yaitu yang ya url user email phone wkwk haha tak tidak gak ga nggak enggak belum jangan bukan""".split())
data["safe"] = apply(data[TEXT_COL], SAFE); data["embed"] = apply(data.safe, PLACEHOLDER); data["normal"] = apply(data.embed, NORMAL)
bow = apply(data.normal, CLEAN).str.split()
df_bow = pd.Series(Counter(w for s in bow for w in set(s))) / len(bow)
DOMAIN_STOP = set(df_bow[df_bow > DF_MAX].index)
show(f"Stopword domain · df > {DF_MAX:.0%}", df_bow[df_bow > DF_MAX].sort_values(ascending=False).rename("df"))
data["bow"] = bow.map(lambda s: " ".join(w for w in s if w not in STOP | DOMAIN_STOP and len(w) > 2))
data["bow_stem"] = stem_series(data.bow)
from gensim.models.phrases import Phrases
phrases = Phrases(data.bow.str.split().tolist(), min_count=5, threshold=10, delimiter="_")
data["bow_bigram"] = [" ".join(phrases[s]) for s in data.bow.str.split()]
show("Bigram yang terbentuk", pd.Series(Counter(w for s in data.bow_bigram.str.split() for w in s if "_" in w)).nlargest(TOP))


Stopword domain · df > 25%


,df
mbg,0.458
yang,0.455
di,0.320
tidak,0.311
dan,0.302


stemming 4113 kata dari 14130 kata unik



Bigram yang terbentuk


,0
prof_gema,183
pak_gema,144
orang_tua,99
tata_kelola,94
tepat_sasaran,93
lapangan_kerja,82
anak_sekolah,75
program_bagus,69
pak_prabowo,66
rakyat_indonesia,64


In [17]:
views = ["safe", "embed", "normal", "bow", "bow_stem", "bow_bigram"]
show("Audit preprocessing", pd.DataFrame({v: {"kosong_%": 100 * data[v].str.strip().eq("").mean(), "kata_median": data[v].str.split().str.len().median(),
    "kosakata": len(set(" ".join(data[v]).split()))} for v in views}).T)
show("Before/after", data[[TEXT_COL, "bow", "bow_stem", "bow_bigram"]].sample(4, random_state=SEED))


Audit preprocessing


,kosong_%,kata_median,kosakata
safe,0.00,17.0,28086.0
embed,0.00,17.0,28080.0
normal,0.00,17.0,23415.0
bow,0.74,11.0,14130.0
bow_stem,0.74,11.0,12678.0
bow_bigram,0.74,10.0,14482.0



Before/after


,Comment,bow,bow_stem,bow_bigram
1501,Teori,teori,teori,teori
2586,harusnya prabowo punya prinsip.sendiri.janga dengar tekanan jokowiwik. engga guna nya masukan jelek.kensel...,harusnya prabowo punya prinsip sendiri janga dengar tekanan jokowiwik engga guna masukan jelek kensel klw ...,harus prabowo punya prinsip sendiri janga dengar tekan jokowiwik engga guna masuk jelek kensel klw wowo ja...,harusnya prabowo punya prinsip sendiri janga dengar tekanan jokowiwik engga guna masukan jelek kensel klw ...
2653,"Banyak permintaan beras, ujung2nya terbuang percuma",permintaan beras ujung2nya terbuang percuma,minta beras ujung2nya buang percuma,permintaan beras ujung2nya terbuang percuma
1055,Kenapa mbg tidak dikelola lewat kmp?,dikelola lewat kmp,kelola lewat kmp,dikelola lewat kmp


## 05 · Fungsi evaluasi tanpa label

Kriteria di sini **tidak memakai label**. Semua model menghasilkan dua hal yang sama: daftar kata kunci per topik dan topik dominan setiap dokumen. Dengan begitu, model yang cara kerjanya berbeda tetap bisa dibandingkan.
- **NPMI** dihitung dengan co-occurrence dokumen, pada korpus versi teks yang dipakai model itu sendiri.
- **Stabilitas:** model di-fit dengan 3 seed. Topik antar seed dipasangkan dengan Hungarian matching, lalu Jaccard top-10 kata dirata-rata.

In [18]:
REF = {}
def reference(view):
    if view not in REF:
        vectorizer = CountVectorizer(binary=True, min_df=MIN_DF, token_pattern=r"\S+").fit(data[view])
        matrix = vectorizer.transform(data[view]).tocsc()
        REF[view] = (matrix, vectorizer.vocabulary_, np.asarray(matrix.sum(0)).ravel())
    return REF[view]

def npmi(words, view):
    matrix, vocab, docfreq = reference(view); n = matrix.shape[0]
    ids = [vocab[w] for w in words if w in vocab]; scores = []
    for a, b in itertools.combinations(ids, 2):
        both = matrix[:, a].multiply(matrix[:, b]).sum() / n
        scores.append(-1. if both == 0 else np.log(both / (docfreq[a] * docfreq[b] / n ** 2)) / -np.log(both))
    return float(np.mean(scores)) if scores else np.nan

def ctfidf(labels, view, n=25):
    texts = data[view].to_numpy(); topics = [c for c in np.unique(labels) if c >= 0]
    joined = [" ".join(texts[labels == c]) for c in topics]
    vectorizer = CountVectorizer(token_pattern=r"\S+").fit(joined); tf = vectorizer.transform(joined).toarray().astype(float)
    weight = tf * np.log1p(tf.sum() / len(topics) / np.maximum(tf.sum(0), 1)); names = vectorizer.get_feature_names_out()
    return {c: list(names[np.argsort(-weight[i])[:n]]) for i, c in enumerate(topics)}

def match_stability(runs):
    """runs: list of {topik: [kata]} dari seed berbeda. Rata-rata Jaccard top-10 setelah Hungarian matching."""
    scores = []
    for a, b in itertools.combinations(runs, 2):
        A, B = list(a.values()), list(b.values())
        sim = np.array([[len(set(x[:10]) & set(y[:10])) / len(set(x[:10]) | set(y[:10])) for y in B] for x in A])
        r, c = linear_sum_assignment(-sim); scores.append(sim[r, c].sum() / max(len(A), len(B)))
    return float(np.mean(scores))

In [19]:
def audit_scores(labels):
    truth = data["_audit"].to_numpy(); valid = labels >= 0
    table = pd.crosstab(labels, truth).to_numpy(); r, c = linear_sum_assignment(-table)
    return {"audit_NMI": normalized_mutual_info_score(truth, labels), "audit_ARI": adjusted_rand_score(truth, labels),
            "audit_NMI_tanpa_outlier": normalized_mutual_info_score(truth[valid], labels[valid]),
            "audit_acc_hungarian": table[r, c].sum() / len(labels), "audit_purity": table.max(1).sum() / len(labels)}

rows, STORE = [], {}
def evaluate(name, topics, labels, view, **extra):
    """topics: {topik: [kata]}; labels: topik dominan tiap dokumen (-1 = outlier)."""
    labels = np.asarray(labels); valid = labels >= 0
    share = pd.Series(labels[valid]).value_counts(normalize=True) * valid.mean()
    per_topic = {c: npmi(ws[:TOP], view) for c, ws in topics.items()}
    top25 = [w for ws in topics.values() for w in ws[:25]]
    coherence = np.nanmean(list(per_topic.values())); diversity = len(set(top25)) / max(len(top25), 1)
    row = {"model": name, "view": view, **extra, "k": len(topics), "outlier_%": 100 * (~valid).mean(),
           "max_share_%": 100 * share.max() if len(share) else np.nan, "NPMI": coherence, "diversity": diversity,
           "TQ": (coherence + 1) / 2 * diversity}
    if AUDIT_COL: row.update(audit_scores(labels))
    rows.append(row); STORE[name] = {"topics": topics, "labels": labels, "view": view, "per_topic_npmi": per_topic}
    return row
print("Fungsi evaluasi siap.")

Fungsi evaluasi siap.


## 06 · Representasi & ablasi preprocessing

e5 dihitung sekali (frozen, mean pooling, awalan `query: `). Ablasi preprocessing memakai NMF, karena paling cepat dan paling sensitif terhadap pilihan kata. Versi BoW dipilih dari TQ, bukan dari audit.

In [20]:
def locate(name, hf_id):
    found = list(INPUT.glob(f"**/models/{name}/config.json")); return str(found[0].parent) if found else hf_id

def embed(model_id, texts):
    from transformers import AutoTokenizer, AutoModel
    tokenizer = AutoTokenizer.from_pretrained(model_id); model = AutoModel.from_pretrained(model_id).cuda().half().eval()
    order = np.argsort([len(x) for x in texts]); parts = []
    with torch.inference_mode():
        for i in range(0, len(texts), BATCH):
            enc = tokenizer([texts[j] for j in order[i:i + BATCH]], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to("cuda")
            hidden = model(**enc).last_hidden_state; mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            parts.append(((hidden * mask).sum(1) / mask.sum(1)).float().cpu().numpy())
    vectors = np.zeros((len(texts), parts[0].shape[1]), "float32"); vectors[order] = np.concatenate(parts)
    del model; torch.cuda.empty_cache(); return normalize(vectors)

start = time.perf_counter()
E5 = embed(locate("e5", "intfloat/multilingual-e5-base"), ("query: " + data.embed).tolist())
print(f"e5: {E5.shape} dalam {time.perf_counter() - start:.0f} detik")
def tfidf(view): v = TfidfVectorizer(min_df=MIN_DF, max_df=.5, token_pattern=r"\S+", sublinear_tf=True); return v, v.fit_transform(data[view])
def counts_of(view): v = CountVectorizer(min_df=MIN_DF, max_df=.5, token_pattern=r"\S+"); return v, v.fit_transform(data[view])
def factor_topics(components, names, n=25): return {i: list(names[np.argsort(-row)[:n]]) for i, row in enumerate(components)}

e5: (5000, 768) dalam 13 detik


In [21]:
K_REF = K_GRID[len(K_GRID) // 2]; ablation = []
for view in ["normal", "bow", "bow_stem", "bow_bigram"]:
    vectorizer, X = tfidf(view)
    model = NMF(K_REF, init="nndsvda", max_iter=400, random_state=SEED); W = model.fit_transform(X)
    labels = np.where(W.max(1) > 0, W.argmax(1), -1)
    ablation.append({"view": view, "vocab": X.shape[1], **evaluate(f"ablasi_{view}", factor_topics(model.components_, vectorizer.get_feature_names_out()), labels, view, algo="ablasi")})
rows = [r for r in rows if r.get("algo") != "ablasi"]
ablation = pd.DataFrame(ablation).drop(columns=["model", "algo"])
show(f"6.1 Ablasi preprocessing · NMF k={K_REF}", ablation)
BOW_VIEW = ablation.sort_values("TQ").iloc[-1]["view"]
print("Versi BoW terpilih (TQ tertinggi):", BOW_VIEW)


6.1 Ablasi preprocessing · NMF k=12


,view,vocab,k,outlier_%,max_share_%,NPMI,diversity,TQ
0,normal,4844,12,1.62,17.78,0.040,0.683,0.355
1,bow,4124,12,1.80,21.70,0.058,0.737,0.390
2,bow_stem,2760,12,1.80,23.50,0.141,0.700,0.399
3,bow_bigram,4441,12,1.80,44.02,-0.056,0.720,0.340


Versi BoW terpilih (TQ tertinggi): bow_stem


## 07 · Model: grid k untuk setiap model

Setiap model dicoba pada semua `K_GRID` dengan 3 seed. Baris yang dicatat adalah seed pertama, dan stabilitas dihitung dari ketiga seed. HDBSCAN tidak memakai k, jadi yang divariasikan adalah `min_cluster_size`.

In [22]:
tfidf_vec, X_tfidf = tfidf(BOW_VIEW); count_vec, X_count = counts_of(BOW_VIEW)
names_tfidf, names_count = tfidf_vec.get_feature_names_out(), count_vec.get_feature_names_out()
THETA = {}

def fit_once(algo, k, seed):
    if algo == "tfidf_kmeans":
        labels = KMeans(k, n_init=1, random_state=seed).fit_predict(X_tfidf); return ctfidf(labels, BOW_VIEW), labels, None
    if algo == "e5_kmeans":
        labels = KMeans(k, n_init=1, random_state=seed).fit_predict(E5); return ctfidf(labels, BOW_VIEW), labels, None
    if algo == "lda":
        model = LatentDirichletAllocation(k, learning_method="batch", max_iter=25, random_state=seed); theta = model.fit_transform(X_count)
        return factor_topics(model.components_, names_count), theta.argmax(1), theta
    model = NMF(k, init="nndsvda" if seed == SEED else "random", max_iter=400, random_state=seed); W = model.fit_transform(X_tfidf)
    theta = W / np.maximum(W.sum(1, keepdims=True), 1e-12)
    return factor_topics(model.components_, names_tfidf), np.where(W.max(1) > 0, W.argmax(1), -1), theta

start = time.perf_counter()
for algo in ["tfidf_kmeans", "lda", "nmf", "e5_kmeans"]:
    for k in K_GRID:
        fits = [fit_once(algo, k, SEED + s) for s in range(SEEDS)]
        evaluate(f"{algo}_k{k}", fits[0][0], fits[0][1], BOW_VIEW, algo=algo, stability=match_stability([f[0] for f in fits]))
        THETA[f"{algo}_k{k}"] = fits[0][2]
print(f"grid selesai dalam {time.perf_counter() - start:.0f} detik")

grid selesai dalam 389 detik


In [23]:
import umap
reductions = [umap.UMAP(n_neighbors=15, n_components=5, min_dist=0., metric="cosine", random_state=SEED + s).fit_transform(E5) for s in range(SEEDS)]
for size in [max(10, len(data) // 200), max(15, len(data) // 100), max(25, len(data) // 50)]:
    fits = []
    for U in reductions:
        labels = HDBSCAN(min_cluster_size=size, min_samples=5).fit_predict(U); fits.append((ctfidf(labels, BOW_VIEW), labels))
    evaluate(f"bertopic_mcs{size}", fits[0][0], fits[0][1], BOW_VIEW, algo="bertopic", stability=match_stability([f[0] for f in fits]), min_cluster_size=size)
rng = np.random.default_rng(SEED); vocab_pool = np.array(names_tfidf)
random_topics = {i: list(rng.choice(vocab_pool, 25, replace=False)) for i in range(10)}
evaluate("acak_k10", random_topics, rng.integers(0, 10, len(data)), BOW_VIEW, algo="acak", stability=0.)
board = pd.DataFrame(rows)
show("Grid · tanpa kolom audit", board.drop(columns=[c for c in board if c.startswith("audit_")]).sort_values("TQ", ascending=False).head(15))

I0000 00:00:1794163029.149760     142 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.



Grid · tanpa kolom audit


,model,view,algo,stability,k,outlier_%,max_share_%,NPMI,diversity,TQ,min_cluster_size
0,tfidf_kmeans_k5,bow_stem,tfidf_kmeans,0.318,5,0.0,87.50,0.262,0.752,0.475,NaN
12,nmf_k5,bow_stem,nmf,0.719,5,1.8,36.64,0.126,0.840,0.473,NaN
3,tfidf_kmeans_k12,bow_stem,tfidf_kmeans,0.157,12,0.0,82.54,0.182,0.773,0.457,NaN
2,tfidf_kmeans_k10,bow_stem,tfidf_kmeans,0.117,10,0.0,83.08,0.197,0.756,0.452,NaN
13,nmf_k8,bow_stem,nmf,0.656,8,1.8,29.44,0.146,0.770,0.441,NaN
7,lda_k8,bow_stem,lda,0.421,8,0.0,17.96,0.177,0.745,0.439,NaN
1,tfidf_kmeans_k8,bow_stem,tfidf_kmeans,0.142,8,0.0,83.24,0.169,0.745,0.435,NaN
6,lda_k5,bow_stem,lda,0.407,5,0.0,31.32,0.157,0.728,0.421,NaN
14,nmf_k10,bow_stem,nmf,0.603,10,1.8,23.82,0.136,0.720,0.409,NaN
4,tfidf_kmeans_k15,bow_stem,tfidf_kmeans,0.134,15,0.0,81.12,0.188,0.677,0.402,NaN


## 08 · Pemilihan tanpa label & C_V

**Aturan:** topik raksasa (`max_share_% > MAX_SHARE`) dan outlier berlebihan (`outlier_% > MAX_OUTLIER`) dilarang. Kandidat yang tersisa diurutkan dengan rata-rata peringkat **TQ** dan **stabilitas**. Untuk 5 kandidat teratas dihitung juga **C_V** (gensim, sliding window), yaitu coherence yang paling sering dilaporkan di makalah. C_V hanya dilaporkan, tidak dipakai memilih.

In [24]:
pool = board[(board.algo != "acak") & (board["max_share_%"] <= 100 * MAX_SHARE) & (board["outlier_%"] <= 100 * MAX_OUTLIER)].copy()
pool = pool if len(pool) else board[board.algo != "acak"].copy()
pool["skor_gabungan"] = pool[["TQ", "stability"]].rank(pct=True).mean(axis=1)
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
tokens = data[BOW_VIEW].str.split().tolist(); dictionary = Dictionary(tokens)
def c_v(name):
    topics = [[w for w in ws[:TOP] if w in dictionary.token2id] for ws in STORE[name]["topics"].values()]
    return CoherenceModel(topics=[t for t in topics if len(t) >= 2], texts=tokens, dictionary=dictionary, coherence="c_v", processes=1).get_coherence()
top5 = pool.sort_values("skor_gabungan", ascending=False).head(5).copy()
top5["C_V"] = [c_v(m) for m in top5.model]
show("5 kandidat teratas menurut aturan", top5[["model", "algo", "k", "TQ", "NPMI", "diversity", "stability", "C_V", "max_share_%", "outlier_%", "skor_gabungan"]])
FINAL_BASE = top5.iloc[0].model
show("Terbaik per model · menurut skor gabungan", pool.sort_values("skor_gabungan", ascending=False).groupby("algo").head(1)[["model", "k", "TQ", "stability", "skor_gabungan"]])
print("TERPILIH:", FINAL_BASE)


5 kandidat teratas menurut aturan


,model,algo,k,TQ,NPMI,diversity,stability,C_V,max_share_%,outlier_%,skor_gabungan
12,nmf_k5,nmf,5,0.473,0.126,0.840,0.719,0.490,36.64,1.8,0.944
13,nmf_k8,nmf,8,0.441,0.146,0.770,0.656,0.502,29.44,1.8,0.889
15,nmf_k12,nmf,12,0.399,0.141,0.700,0.646,0.482,23.50,1.8,0.750
14,nmf_k10,nmf,10,0.409,0.136,0.720,0.603,0.490,23.82,1.8,0.750
18,e5_kmeans_k5,e5_kmeans,5,0.265,0.070,0.496,0.976,0.434,28.64,0.0,0.667



Terbaik per model · menurut skor gabungan


,model,k,TQ,stability,skor_gabungan
12,nmf_k5,5,0.473,0.719,0.944
18,e5_kmeans_k5,5,0.265,0.976,0.667
7,lda_k8,8,0.439,0.421,0.611


TERPILIH: nmf_k5


## 09 · Post-processing

Diterapkan pada model terpilih, dan setiap langkah dinilai ulang.
- **Outlier ke topik terdekat:** kalau ada outlier (HDBSCAN), dokumen outlier diarahkan ke topik dengan centroid e5 terdekat, supaya semua dokumen punya topik.
- **Gabung topik kembar:** pasangan topik dengan Jaccard top-10 kata ≥ `MERGE_JACCARD` digabung, karena topik yang hampir sama membingungkan saat presentasi.
- **Gabung topik kecil:** topik di bawah `MIN_TOPIC_SHARE` digabung ke topik dengan centroid e5 terdekat.
- **Label topik:** kata kunci dihitung ulang dengan c-TF-IDF dari dokumen anggotanya, lalu 3 kata teratas dijadikan label.

In [25]:
def to_nearest(labels, members):
    keep = [c for c in np.unique(labels) if c >= 0 and c not in members]
    centers = normalize(np.vstack([E5[labels == c].mean(0) for c in keep]))
    moved = labels.copy(); mask = np.isin(labels, list(members)) | (labels < 0)
    if mask.any(): moved[mask] = np.array(keep)[(E5[mask] @ centers.T).argmax(1)]
    return moved

labels = STORE[FINAL_BASE]["labels"].copy(); topics = STORE[FINAL_BASE]["topics"]; steps = []
if (labels < 0).any():
    labels = to_nearest(labels, set()); evaluate("post_outlier", ctfidf(labels, BOW_VIEW), labels, BOW_VIEW, algo="post"); steps.append("outlier")
merged_pairs = []
for a, b in itertools.combinations(sorted(topics), 2):
    A, B = set(topics[a][:10]), set(topics[b][:10])
    if len(A & B) / len(A | B) >= MERGE_JACCARD: merged_pairs.append((a, b))
for a, b in merged_pairs:
    small, large = sorted([a, b], key=lambda c: (labels == c).sum())
    labels[labels == small] = large
if merged_pairs:
    evaluate("post_gabung_kembar", ctfidf(labels, BOW_VIEW), labels, BOW_VIEW, algo="post"); steps.append(f"gabung {len(merged_pairs)} pasangan")
share = pd.Series(labels).value_counts(normalize=True)
if (share < MIN_TOPIC_SHARE).any():
    labels = to_nearest(labels, set(share[share < MIN_TOPIC_SHARE].index)); steps.append(f"gabung {int((share < MIN_TOPIC_SHARE).sum())} topik kecil")
labels = pd.Series(labels).map({c: i for i, c in enumerate(sorted(np.unique(labels)))}).to_numpy()
final_topics = ctfidf(labels, BOW_VIEW)
evaluate("FINAL", final_topics, labels, BOW_VIEW, algo="final")
board = pd.DataFrame(rows)
show("Post-processing · tanpa audit", board[board.model.isin([FINAL_BASE, "post_outlier", "post_gabung_kembar", "FINAL"])].drop(columns=[c for c in board if c.startswith("audit_")]))
print("Langkah:", steps or "tidak ada yang perlu", "| pasangan kembar:", merged_pairs)


Post-processing · tanpa audit


,model,view,algo,stability,k,outlier_%,max_share_%,NPMI,diversity,TQ,min_cluster_size
12,nmf_k5,bow_stem,nmf,0.719,5,1.8,36.64,0.126,0.840,0.473,NaN
28,post_outlier,bow_stem,post,NaN,5,0.0,36.64,0.152,0.736,0.424,NaN
29,FINAL,bow_stem,final,NaN,5,0.0,36.64,0.152,0.736,0.424,NaN


Langkah: ['outlier'] | pasangan kembar: []


## 10 · Evaluasi & kelemahan

Kolom audit dibuka mulai di sini, tanpa mengubah keputusan di atas. Bagian ini menjawab tiga pertanyaan: seberapa jauh pilihan tanpa label dari yang terbaik, kriteria tanpa label mana yang searah dengan kebenaran, dan topik mana yang lemah.

In [26]:
audit_cols = [c for c in board if c.startswith("audit_")]
cols = ["model", "algo", "k", "TQ", "NPMI", "diversity", "stability", "max_share_%", "outlier_%"] + audit_cols
show("10.1 Semua kandidat", board[[c for c in cols if c in board]].sort_values(audit_cols[0] if audit_cols else "TQ", ascending=False))
if AUDIT_COL:
    ranked = board[~board.algo.isin(["acak", "post"])].sort_values("audit_NMI", ascending=False).reset_index(drop=True)
    final_nmi = board.loc[board.model.eq("FINAL"), "audit_NMI"].iloc[0]
    print(f"FINAL: NMI {final_nmi:.3f}, peringkat {int((ranked.audit_NMI > final_nmi).sum()) + 1} dari {len(ranked)} | terbaik dengan label: "
          f"{ranked.model[0]} NMI {ranked.audit_NMI[0]:.3f} | acak {board.loc[board.model.eq('acak_k10'), 'audit_NMI'].iloc[0]:.3f}")
    grid = board[board.algo.isin(["tfidf_kmeans", "lda", "nmf", "e5_kmeans", "bertopic"])]
    trust = pd.DataFrame({m: spearmanr(grid[m], grid.audit_NMI, nan_policy="omit") for m in ["TQ", "NPMI", "diversity", "stability", "max_share_%", "k"]},
                         index=["spearman", "p"]).T
    show("10.2 Kriteria tanpa label vs NMI audit · positif = bisa dipercaya", trust.sort_values("spearman", ascending=False))
    show("NMI terbaik per model", grid.groupby("algo").audit_NMI.max().sort_values(ascending=False))


10.1 Semua kandidat


,model,algo,k,TQ,NPMI,diversity,stability,max_share_%,outlier_%
0,tfidf_kmeans_k5,tfidf_kmeans,5,0.475,0.262,0.752,0.318,87.50,0.00
12,nmf_k5,nmf,5,0.473,0.126,0.840,0.719,36.64,1.80
3,tfidf_kmeans_k12,tfidf_kmeans,12,0.457,0.182,0.773,0.157,82.54,0.00
2,tfidf_kmeans_k10,tfidf_kmeans,10,0.452,0.197,0.756,0.117,83.08,0.00
13,nmf_k8,nmf,8,0.441,0.146,0.770,0.656,29.44,1.80
7,lda_k8,lda,8,0.439,0.177,0.745,0.421,17.96,0.00
1,tfidf_kmeans_k8,tfidf_kmeans,8,0.435,0.169,0.745,0.142,83.24,0.00
29,FINAL,final,5,0.424,0.152,0.736,NaN,36.64,0.00
28,post_outlier,post,5,0.424,0.152,0.736,NaN,36.64,0.00
6,lda_k5,lda,5,0.421,0.157,0.728,0.407,31.32,0.00


In [27]:
vectors = normalize(np.vstack([E5[labels == c].mean(0) for c in sorted(final_topics)]))
cosine = (E5 * vectors[labels]).sum(1)
docfreq_ref = reference(BOW_VIEW)
generic = {w for w, d in zip(*[list(docfreq_ref[1].keys()), docfreq_ref[2][list(docfreq_ref[1].values())] / len(data)]) if d > .05}
dup_key = data.normal.str.replace(r"\s+", " ", regex=True).str.strip()
cards = []
for c, ws in sorted(final_topics.items()):
    m = labels == c
    row = {"topik": c, "label": " / ".join(ws[:3]), "n": int(m.sum()), "%": 100 * m.mean(), "kata_kunci": ", ".join(ws[:TOP]),
           "NPMI": npmi(ws[:TOP], BOW_VIEW), "kata_generik_%": 100 * np.mean([w in generic for w in ws[:TOP]]),
           "porsi_duplikat_%": 100 * dup_key[m].duplicated(keep=False).mean(), "kemiripan_ke_pusat": cosine[m].mean(),
           "contoh": " | ".join(data.loc[m].assign(s=cosine[m]).nlargest(2, "s")[TEXT_COL].str.slice(0, 70))}
    if AUDIT_COL:
        audit = pd.Series(data["_audit"].to_numpy()[m]).value_counts(normalize=True)
        row.update(audit_mayoritas=audit.index[0], audit_purity=audit.iloc[0])
    cards.append(row)
cards = pd.DataFrame(cards)
show("10.3 Kartu topik · urut dari NPMI terendah", cards.sort_values("NPMI"))


10.3 Kartu topik · urut dari NPMI terendah


,topik,label,n,%,kata_kunci,NPMI,kata_generik_%,porsi_duplikat_%,kemiripan_ke_pusat,contoh
4,4,mantap / prof / gema,211,4.22,"mantap, prof, gema, bang, jelas, bos, pak, cerah, orang, data",0.065,50.0,7.109,0.902,Mantap ❤❤🙏 | mantap...terima kasih infonya...❤❤❤❤
1,1,prof / gema / pak,664,13.28,"prof, gema, pak, cerah, data, jelas, kasih, anda, terus, terima",0.146,40.0,1.355,0.911,Terima kasih prof.gema atas pemaparan nya 🙏 | Suka sekali penjelasan prof Gema..jelas lugas dgn data yg ak...
2,2,korupsi / rakyat / program,1623,32.46,"korupsi, rakyat, program, negara, presiden, indonesia, prabowo, hukum, koruptor, perintah",0.180,80.0,1.540,0.913,MBG menurutku bagus bermanfaat yg tdk bagus adalah org2 yg melakukan | Kalau masih ada mbg jangan harap ...
3,3,baik / kelola / tuju,670,13.40,"baik, kelola, tuju, lanjut, tata, program, henti, perlu, benar, perintah",0.180,50.0,1.642,0.913,"mbg adalah perbuatan yg mulia bermanfaat bagi orang yg tidak mampu, ha | Aku pendukung MBG Krn sangat memb..."
0,0,makan / anak / sekolah,1832,36.64,"makan, anak, sekolah, orang, kerja, harga, dapur, sppg, uang, tani",0.190,90.0,0.873,0.912,"MBG cuma menitipkan uang ke orang2 untuk dibelanjakan dan disalurkan, | MBG itu bagus ,hanya tata kelolan..."


In [28]:
flags = pd.DataFrame({"topik": cards.topik, "label": cards.label, "NPMI_negatif": cards.NPMI < 0,
    "kata_generik_tinggi": cards["kata_generik_%"] >= 40, "didominasi_duplikat": cards["porsi_duplikat_%"] >= 30,
    "raksasa": cards["%"] > 100 * MAX_SHARE, "kecil": cards["%"] < 2})
show("10.4 Bendera kelemahan per topik", flags[flags.drop(columns=["topik", "label"]).any(axis=1)])
sim = vectors @ vectors.T; np.fill_diagonal(sim, -1); pairs = np.dstack(np.unravel_index(np.argsort(-sim, axis=None), sim.shape))[0]
close = [(int(a), int(b), sim[a, b]) for a, b in pairs if a < b][:5]
show("10.5 Pasangan topik paling mirip (centroid e5) · kandidat digabung", pd.DataFrame(close, columns=["topik_a", "topik_b", "cosine"]).assign(
    label_a=lambda f: f.topik_a.map(cards.set_index("topik").label), label_b=lambda f: f.topik_b.map(cards.set_index("topik").label)))
length = pd.cut(data[BOW_VIEW].str.split().str.len(), [-1, 0, 3, 6, 12, np.inf], labels=["kosong", "1-3", "4-6", "7-12", ">12"])
slice_rows = []
for value, ids in pd.Series(np.arange(len(data))).groupby(length.to_numpy(), observed=True):
    ids = ids.to_numpy(); row = {"panjang_bow": value, "n": len(ids), "kemiripan_ke_pusat": cosine[ids].mean()}
    if AUDIT_COL: row["NMI_slice"] = normalized_mutual_info_score(data["_audit"].to_numpy()[ids], labels[ids])
    slice_rows.append(row)
show("10.6 Kelemahan menurut panjang dokumen", pd.DataFrame(slice_rows))
base_theta = THETA.get(FINAL_BASE)
if base_theta is not None:
    print(f"10.7 Dokumen ambigu (proporsi topik dominan < 0.4): {100 * (base_theta.max(1) < .4).mean():.1f}%")
show("10.8 Paling jauh dari pusat topik · kandidat salah topik", data[[TEXT_COL]].assign(topik=labels, kemiripan=cosine).nsmallest(6, "kemiripan"))


10.4 Bendera kelemahan per topik


,topik,label,NPMI_negatif,kata_generik_tinggi,didominasi_duplikat,raksasa,kecil
0,0,makan / anak / sekolah,False,True,False,False,False
1,1,prof / gema / pak,False,True,False,False,False
2,2,korupsi / rakyat / program,False,True,False,False,False
3,3,baik / kelola / tuju,False,True,False,False,False
4,4,mantap / prof / gema,False,True,False,False,False



10.5 Pasangan topik paling mirip (centroid e5) · kandidat digabung


,topik_a,topik_b,cosine,label_a,label_b
0,2,3,0.995,korupsi / rakyat / program,baik / kelola / tuju
1,0,2,0.994,makan / anak / sekolah,korupsi / rakyat / program
2,0,3,0.994,makan / anak / sekolah,baik / kelola / tuju
3,1,3,0.989,prof / gema / pak,baik / kelola / tuju
4,1,4,0.988,prof / gema / pak,mantap / prof / gema



10.6 Kelemahan menurut panjang dokumen


,panjang_bow,n,kemiripan_ke_pusat
0,1-3,616,0.895
1,4-6,845,0.906
2,7-12,1268,0.913
3,>12,2234,0.918
4,kosong,37,0.906


10.7 Dokumen ambigu (proporsi topik dominan < 0.4): 7.2%

10.8 Paling jauh dari pusat topik · kandidat salah topik


,Comment,topik,kemiripan
1009,Aamiin,3,0.806
1239,Cerdas,1,0.812
1919,Mantapp,1,0.817
1792,Kepuasan 58 persen dari mana?,0,0.821
4779,Buah apa yang paling manis?,0,0.823
3671,Bazzer,4,0.825


## 11 · Keluaran

File `topics.csv` berisi topik dan label untuk setiap dokumen. `kartu_topik.csv` berisi ringkasan topik untuk presentasi.

In [29]:
data[["row_id", TEXT_COL]].assign(topik=labels, label=[cards.set_index("topik").label[c] for c in labels], kemiripan=cosine).to_csv(OUT / "topics.csv", index=False)
board.to_csv(OUT / "kandidat.csv", index=False); cards.to_csv(OUT / "kartu_topik.csv", index=False); ablation.to_csv(OUT / "ablasi_preprocessing.csv", index=False)
import importlib.metadata as metadata
manifest = {"dataset": DATASET, "rows": len(data), "bow_view": BOW_VIEW, "base": FINAL_BASE, "k_final": len(final_topics), "post": steps,
    "final": {k: v for k, v in board.loc[board.model.eq("FINAL")].iloc[0].items() if isinstance(v, (int, float, np.floating))},
    "seconds": time.perf_counter() - RUN_START, "gpu": torch.cuda.get_device_name(0),
    "versions": {p: metadata.version(p) for p in ["torch", "transformers", "scikit-learn", "umap-learn", "gensim", "pandas"]}}
(OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=float))
print(json.dumps(manifest, indent=2, default=float))

{
  "dataset": "mbg",
  "rows": 5000,
  "bow_view": "bow_stem",
  "base": "nmf_k5",
  "k_final": 5,
  "post": [
    "outlier"
  ],
  "final": {
    "stability": NaN,
    "outlier_%": 0.0,
    "max_share_%": 36.64,
    "NPMI": 0.15215759910008017,
    "diversity": 0.736,
    "TQ": 0.42399399646882946,
    "min_cluster_size": NaN
  },
  "seconds": 555.275066764,
  "gpu": "Tesla T4",
  "versions": {
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2",
    "scikit-learn": "1.6.1",
    "umap-learn": "0.5.12",
    "gensim": "4.4.0",
    "pandas": "2.3.3"
  }
}


## Sumber & batas interpretasi
- Preprocessing diambil dari `preprocessing_universal/stepwise.py`. Stopword tambahan dan stopword domain bergantung pada data, jadi periksa tabel 4.1.
- NPMI mengikuti Röder dkk. 2015, dan C_V memakai implementasi gensim. Topic quality = coherence × diversity mengikuti Dieng dkk. 2020 (ETM), dengan NPMI dinormalisasi ke 0–1.
- c-TF-IDF dan pipeline e5 + UMAP + HDBSCAN mengikuti ide [BERTopic](https://maartengr.github.io/BERTopic/), diimplementasikan langsung tanpa library BERTopic.
- Kategori berita bukan topik "benar", karena satu kategori bisa berisi beberapa topik. NMI audit hanya sinyal kasar.
- Satu sampel 5.000 dokumen. Stabilitas mengukur variasi seed, bukan variasi sampel data.